In [1]:
# Import Google Drive access library from Google Colab
# This allows the notebook to access files stored in your Google Drive
from google.colab import drive

# Mount (connect) Google Drive so the program can read model and image files from it
drive.mount('/content/drive')


# Import operating system utilities
# Helps work with file paths and directories
import os

# Import glob library
# Used to search for files in a folder that match a specific pattern (like all .jpg images)
import glob

# Import NumPy library for numerical operations
# Used for handling image data as arrays
import numpy as np

# Import function to load the saved trained model
# This model was created during the training phase
from tensorflow.keras.models import load_model

# Import image processing utilities
# These help load images and convert them into a format suitable for the model
from tensorflow.keras.preprocessing import image

# Import the same preprocessing function used during training
# Ensures the input images are formatted exactly like training images
from tensorflow.keras.applications.resnet50 import preprocess_input


# Define the folder path that contains test images
# These are shrimp images that the model will analyze
test_images_path = '/content/drive/MyDrive/ML_WSSV/ML_WSSV/WSSV/images'


# -----------------------------
# Load the Trained Model
# -----------------------------

# Load the previously trained ResNet50 model from Google Drive
# This model already learned how to identify Healthy and WSSV shrimp
model = load_model('/content/drive/MyDrive/ML_WSSV/WSSV/Trained_Model/resnet50_model.keras')

# Print confirmation message so the user knows the model loaded correctly
print("Model loaded successfully.")


# -----------------------------
# Function to Predict Shrimp Health
# -----------------------------

# Define a function that takes an image file path and the trained model
# It will analyze the image and return the predicted class
def predict_image_class(image_path, model):

    # Load the image from the given path
    # Resize it to 224x224 pixels because ResNet50 expects this size
    img = image.load_img(image_path, target_size=(224, 224))

    # Convert the image into a numeric array
    # Machine learning models work with numbers rather than image files
    img_array = image.img_to_array(img)

    # Expand dimensions to simulate a batch of images
    # Even a single image must be shaped like a batch for the model
    img_array = np.expand_dims(img_array, axis=0)

    # Apply the same preprocessing used during training
    # This ensures the model receives images in the expected format
    img_array = preprocess_input(img_array)


    # Use the trained model to make a prediction
    # The model outputs a probability value between 0 and 1
    prediction = model.predict(img_array)[0][0]


    # Determine the predicted class based on probability
    # If the value is greater than 0.5 → classify as WSSV
    # Otherwise → classify as Healthy
    class_label = 'WSSV' if prediction > 0.5 else 'HEALTHY'


    # Calculate confidence score
    # If predicted WSSV → use the prediction value
    # If predicted Healthy → use the inverse probability
    confidence = prediction if prediction > 0.5 else 1 - prediction


    # Return predicted class and confidence percentage
    return class_label, round(confidence * 100, 2)


# Print message indicating the folder scanning has started
print("📁 Scanning folder:", test_images_path)


# Define common image file formats to search for
# This allows the program to detect multiple image types
image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp']


# Loop through each file type extension
for ext in image_extensions:

    # Search for all image files in the folder with that extension
    for img_path in glob.glob(os.path.join(test_images_path, ext)):

        # Run the prediction function on the image
        label, conf = predict_image_class(img_path, model)

        # Print the result showing:
        # - image name
        # - predicted shrimp health status
        # - confidence level of the prediction
        print(f"🖼️ {os.path.basename(img_path)} → Predicted: {label} ({conf}% confidence)")

Mounted at /content/drive
Model loaded successfully.
📁 Scanning folder: /content/drive/MyDrive/ML_WSSV/ML_WSSV/WSSV/images
